In [1]:
%matplotlib inline
%config InlineBackend.figure_format = "retina"
import matplotlib.pyplot as plt
import numpy as np

In [2]:
from cocoa_fisher import FisherMeta, FisherCase
from cocoa_fisher import show_matrix, invert_matrix

real = FisherMeta("real", to_sigma8=True)
fourier = FisherMeta("fourier", to_sigma8=True)

# Data Vectors and Covariance Matrices (Figures 1, 2, 3, and 18)

In [3]:
Oct5_plots = False  # Repurposed on 12/4/2025.

if Oct5_plots:

    real.ticks = np.searchsorted(real.mask, np.cumsum(
        [0, real.n_ss, real.n_ss, real.n_gs, real.n_gg]) * real.n_coord)
    real.labels = [r"$\xi_+ (\theta)$", r"$\xi_- (\theta)$",
                   r"$\gamma_{\rm t} (\theta)$", r"$w (\theta)$"]
    fourier.ticks = np.searchsorted(fourier.mask, np.cumsum(
        [0, fourier.n_ss, fourier.n_gs, fourier.n_gg]) * fourier.n_coord)
    fourier.labels = [r"$C_{\rm ss} (\ell)$", r"$C_{\rm gs} (\ell)$",
                      r"$C_{\rm gg} (\ell)$"]

def add_ticks_and_labels(ax, meta, axes=["x", "y"]):
    for tick in meta.ticks[1:-1]:
        ax.axhline(tick-0.5, color="k", lw=1, ls="--")
        ax.axvline(tick-0.5, color="k", lw=1, ls="--")

    for axis in axes:
        getattr(ax, f"set_{axis}ticks")((meta.ticks[1:]+meta.ticks[:-1])/2-0.5, minor=True)
        getattr(ax, f"set_{axis}ticklabels")(meta.labels, minor=True)
    for axis in ["x", "y"]:
        getattr(ax, f"set_{axis}ticks")(meta.ticks-0.5, labels=[])
        getattr(ax, f"tick_params")(axis=axis, which="minor", length=0)

In [4]:
# Figure 1: Layouts of data vectors
if Oct5_plots:
    fig, axs = plt.subplots(4, 1, figsize=(10.8, 6.0), layout="constrained")

    for j, meta in enumerate([fourier, real]):
        y = meta.datav[meta.mask]; c=[f"C{int(e)}" for e in y < 0]
        yerr = np.sqrt(np.diag(meta.cov_g + meta.cov_ng))[meta.mask]

        axs[j*2].scatter(np.arange(len(y)), np.abs(y), s=1, c=c)
        axs[j*2].set_ylabel("|Data Vector Elem|\n(" + meta.space.capitalize() + ")")
        axs[j*2+1].scatter(np.arange(len(y)), np.abs(y)/yerr, s=1, c=c)
        axs[j*2+1].set_ylabel("|DV Elem| / Error\n(" + meta.space.capitalize() + ")")

        for i in range(2):
            for tick in meta.ticks[1:-1]:
                axs[j*2+i].axvline(tick-0.5, color="k", lw=1, ls="-")
            for t in range(7):  # tomographic bin
                for s in [0] if meta.space == "fourier" else [0, 540]:
                    axs[j*2+i].axvline(s+sum(range(8, 7-t, -1))*15-0.5,
                                    color="k", lw=1, ls="--")

            for p, pair in enumerate(meta.pairs_gs):
                if p > 0 and pair[0] != meta.pairs_gs[p-1][0]:
                    axs[j*2+i].axvline(np.searchsorted(meta.mask, meta.ticks[-3]+p*15)-0.5,
                                    color="k", lw=1, ls="--")

        axs[j*2].set_xticks([]); axs[j*2].set_xticklabels([])
        axs[j*2+1].set_ylim(bottom=0.1, top=(np.abs(y)/yerr).max())
        for i in range(2):
            axs[j*2+i].set_yscale("log")
            axs[j*2+i].set_xlim((-5.5, len(y)+4.5))

    ax = axs[0]; ymin = np.abs(fourier.datav[fourier.mask]).min()
    for t in range(3):
        ax.text(sum(range(8, 8-t, -1))*15, ymin,
                r"$C_{\rm ss} (\ell)$"+'\n'+rf"$({t+1}, {t+1})$"+'\n'\
                    +rf"$\to ({t+1}, 8)$", ha="left", va="bottom")
    ax.text(315, ymin, r"$\ldots$", ha="left", va="bottom")

    for p, pair in enumerate(fourier.pairs_gs):
        if p == 0 or p > 0 and pair[0] != fourier.pairs_gs[p-1][0]:
            if pair[0] == 3:
                ax.text(np.searchsorted(fourier.mask, fourier.ticks[-3]+p*15), ymin,
                        "$\ldots$", ha="left", va="bottom")
                break
            ax.text(np.searchsorted(fourier.mask, fourier.ticks[-3]+p*15), ymin,
                    r"$C_{\rm gs} (\ell)$"+'\n'+rf"$({pair[0]+1}, {pair[1]+1})$"+'\n'\
                        +rf"$\to ({pair[0]+1}, 8)$", ha="left", va="bottom")

    ax.text(fourier.ticks[-2], ymin,
            r"$C_{\rm gg} (\ell)$"+'\n'+r"$(1, 1)$"+'\n'\
                +rf"$\to (8, 8)$", ha="left", va="bottom")

    ax = axs[2]; ymin = np.abs(real.datav[real.mask]).min()
    for t in range(3):
        ax.text(sum(range(8, 8-t, -1))*15, ymin,
                r"$\xi_+ (\theta)$"+'\n'+rf"$({t+1}, {t+1})$"+'\n'\
                    +rf"$\ \ \to$"+'\n'+rf"$({t+1}, 8)$", ha="left", va="bottom")
        ax.text(540+sum(range(8, 8-t, -1))*15, ymin,
                r"$\xi_- (\theta)$"+'\n'+rf"$({t+1}, {t+1})$"+'\n'\
                    +rf"$\ \ \to$"+'\n'+rf"$({t+1}, 8)$", ha="left", va="bottom")
    for s in [0, 540]:
        ax.text(s+315, ymin, r"$\ldots$", ha="left", va="bottom")

    for p, pair in enumerate(real.pairs_gs):
        if p == 0 or p > 0 and pair[0] != real.pairs_gs[p-1][0]:
            if pair[0] == 3:
                ax.text(np.searchsorted(real.mask, real.ticks[-3]+p*15), ymin,
                        "$\ldots$", ha="left", va="bottom")
                break
            ax.text(np.searchsorted(real.mask, real.ticks[-3]+p*15), ymin,
                    r"$\gamma_{\rm t} (\theta)$"+'\n'+rf"$({pair[0]+1}, {pair[1]+1})$"+'\n'\
                        +rf"$\ \ \to$"+'\n'+rf"$({pair[0]+1}, 8)$", ha="left", va="bottom")

    ax.text(real.ticks[-2], ymin,
            r"$w (\theta)$"+'\n'+r"$(1, 1)$"+'\n'\
                +rf"$\to (8, 8)$", ha="left", va="bottom")

    # fig.tight_layout()
    # plt.show()
    fig.savefig(f"fisher_plots/dv_layout.pdf", bbox_inches="tight")
    plt.close(fig)

In [5]:
# Figures 2 and 3: Covariance Matrices and Their Inverses
if Oct5_plots:
    for meta in [real, fourier]:
        fig, axs = plt.subplots(2, 2, figsize=(9.6, 7.8))

        for j in range(2):
            cov = (meta.cov_g + meta.cov_ng * j)[meta.mask][:, meta.mask]
            inv = invert_matrix(cov)

            ax = axs[j, 0]
            show_matrix(fig, ax, cov, symlog=True, oom=12)
            ax.set_title(f"{meta.space.capitalize()}: Covariance {'(G+NG)' if j else '(G)'}")
            add_ticks_and_labels(ax, meta, axes=["x", "y"])

            ax = axs[j, 1]
            show_matrix(fig, ax, inv, symlog=True, oom=6)
            ax.set_title(f"{meta.space.capitalize()}: Inverse {'(G+NG)' if j else '(G)'}")
            add_ticks_and_labels(ax, meta, axes=["x"])

        # fig.tight_layout()
        # plt.show()
        fig.savefig(f"fisher_plots/covinv_{meta.space}.pdf", bbox_inches="tight")
        plt.close(fig)

In [6]:
# Figure 18: Inversion of Covariance Matrices
if Oct5_plots:
    fig, axs = plt.subplots(2, 2, figsize=(9.6, 8.4), sharex="row", sharey="row")

    for j, meta in enumerate([fourier, real]):
        cov = (meta.cov_g + meta.cov_ng)[meta.mask][:, meta.mask]
        inv_np = np.linalg.inv(cov)
        inv_ct = invert_matrix(cov)
        eye_ = np.eye(len(meta.mask))

        show_matrix(fig, axs[j, 0], inv_np @ cov - eye_, symlog=True, oom=8)
        show_matrix(fig, axs[j, 1], inv_ct @ cov - eye_, symlog=True, oom=5)
        axs[j, 0].set_title(f"{meta.space.capitalize()}: Inv (NP) Cov $-$ Eye")
        axs[j, 1].set_title(f"{meta.space.capitalize()}: Inv (CT) Cov $-$ Eye")

        for i in range(2):
            add_ticks_and_labels(axs[j, i], meta, axes=["x", "y"] if i == 0 else ["x"])

    # fig.tight_layout()
    # plt.show()
    fig.savefig("fisher_plots/invert.pdf", bbox_inches="tight")
    plt.close(fig)

# Corner Plots (Figures 5 and 6, 16 and 17)

In [7]:
# Following Nihar Dalal's plot_nb.ipynb.

import os
cwd = os.getcwd()
from cocoa_dvviz import plot_mu_and_cov
os.chdir(cwd)
from IPython.display import clear_output
clear_output(wait=False)
from getdist import MCSamples, plots

def load_chain(file_name, ignore_rows=0.3):
    data = np.loadtxt(file_name, skiprows=1)
    return data[int(len(data)*ignore_rows):]

# From 2025.07.28 cocoa_fisher.ipynb.
params = ["sigma8", "ns", "h0", "omegab", "omegam",
          "roman_DZ_S3", "roman_DZ_S6", "roman_M3", "roman_M6",
          "roman_A1_1", "roman_A1_2", "roman_B1_3", "roman_B1_6"]
subparams = ["sigma8", "ns", "h0", "omegab", "omegam", "roman_M3", "roman_M6"]

labels = dict(
    omegam=r"$\Omega_{\rm m}$", sigma8=r"$\sigma_8$",
    ns=r"$n_{\rm s}$", omegab=r"$\Omega_{\rm b}$", h0=r"$h_0$",
    **{"roman_B1_"+str(i+1): rf"$b^{i+1}$" for i in range(8)},
    **{"roman_DZ_S"+str(i+1): rf"$\Delta_{{z}}^{i+1}$" for i in range(8)},
    **{"roman_M"+str(i+1): rf"$m_{i+1}$" for i in range(8)},
    roman_A1_1=r"$A_{\rm IA}$", roman_A1_2=r"$\eta_{\rm IA}$"
)

In [8]:
from cobaya.input import load_input_file

truth_fourier = load_input_file(
    "../roman_cpip_data_challenge/data_challenge1_fourier_medium/EXAMPLE_EVALUATE.yaml")\
        ["sampler"]["evaluate"]["override"]
truth_fourier["sigma8"] = 0.8255
truth_fourier["h0"] = truth_fourier["H0"] / 100

truth_real = load_input_file(
    "../roman_cpip_data_challenge/data_challenge1_real_medium/EXAMPLE_EVALUATE_TRUTH.yaml")\
        ["sampler"]["evaluate"]["override"]
truth_real["sigma8"] = 0.775
truth_real["h0"] = truth_real["H0"] / 100

In [9]:
def get_samples(space: str):
    chainstem = f"/fs/ess/PCON0003/nddalal/cocoa/Cocoa/projects/roman_{space}/chains/dc1_medium_3x2"
    chaindata = np.concatenate([load_chain(f"{chainstem}.{i+1}.txt") for i in range(8)])

    with open(f"{chainstem}.1.txt", "r") as f:
        column_names = f.readline().strip().split()
    column_names = column_names[1:]
    column_names[column_names.index("sigma8")] = r"$\sigma_8$"

    idx_H0 = column_names.index("H0")
    column_names[idx_H0] = "h0"
    chaindata[:, idx_H0] /= 100

    samples = MCSamples(samples=chaindata, names=column_names)
    mapidx = np.argmax(chaindata[:, 0])  # map: Maximum a posteriori.
    mapmcmc = MCSamples(samples=chaindata[mapidx:mapidx+1, :], names=column_names)

    return samples, mapmcmc

mcmc_real = get_samples("real")
mcmc_fourier = get_samples("fourier")

/users/PAS2055/kailicao/.conda/envs/cocoa/lib/python3.10/site-packages/getdist/mcsamples.py:529: RuntimeWarning: divide by zero encountered in scalar divide
  mult_max = (self.mean_mult * self.numrows) / min(self.numrows // 2, 500)


In [10]:
from scipy.stats import norm

n_std = 2.29574892889472 ** 0.5  # 68% for 2D

def corner_plots(space: str, mcmc: tuple, useparams: list):
    samples, mapmcmc = mcmc

    if len(useparams) == 13:
        g = plots.get_single_plotter(ratio=1, width_inch=8)
        g.triangle_plot([samples]*4, [r"$\sigma_8$"] + useparams[1:], filled=False,
                        contour_colors=["C0", "C1", "C2", "C4"], contour_ls=["-.", "--", ":", "-"],
                        upper_roots=[mapmcmc], legend_labels=["ML", "MAP", "ML$'$", "MCMC"])
    else:
        g = plots.get_single_plotter(ratio=1, width_inch=6)
        g.triangle_plot([samples]*3, [r"$\sigma_8$"] + useparams[1:], filled=False,
                        contour_colors=["C0", "C1", "C4"], contour_ls=["-.", "--", "-"],
                        upper_roots=[mapmcmc], legend_labels=["ML", "MAP", "MCMC"])

    FisherMeta.POSTERIOR = False
    case_ML = FisherCase(FisherMeta(space, to_sigma8=True))
    case_ML.compute_values_and_errors(store_mats=True)

    FisherMeta.POSTERIOR = True
    case_MAP = FisherCase(FisherMeta(space, to_sigma8=True))
    case_MAP.compute_values_and_errors(store_mats=True)
    globals()[f"casemap_{space}"] = case_MAP

    for j, param_j in enumerate(useparams[:-1]):
        for i, param_i in enumerate(useparams[j+1:], start=j+1):
            ax = g.subplots[j, i]

            plot_mu_and_cov(case_ML, "ML", param_i, param_j, ax,
                            "C0", marker=".", s=1, n_std=n_std, ls="-.", lw=0.5)
            plot_mu_and_cov(case_MAP, "MAP", param_i, param_j, ax,
                            "C1", marker=".", s=1, n_std=n_std, ls="--", lw=0.5)
            if len(useparams) == 13:
                plot_mu_and_cov(case_ML, "MLp", param_i, param_j, ax,
                                "C2", marker=".", s=1, n_std=n_std, ls=":", lw=0.5)

            plot_mu_and_cov(case_MAP, "MAP", param_j, param_i, g.subplots[i, j],
                            "C1", marker=".", s=1, n_std=n_std, ls="--", lw=0.5)

    for j, param_j in enumerate(useparams):
        for i, param_i in enumerate(useparams):
            ax = g.subplots[j, i]

            if j == i:
                ax.axvline(globals()[f"truth_{space}"][param_i], color="grey", lw=0.5, ls=":", zorder=-1)
            else:
                ax.axvline(globals()[f"truth_{space}"][param_i], color="grey", lw=0.5, ls=":", zorder=-1)
                ax.axhline(globals()[f"truth_{space}"][param_j], color="grey", lw=0.5, ls=":", zorder=-1)

    if space == "fourier": x, y = 0.833, 0.323
    elif space == "real": x, y = 0.783, 0.26
    g.subplots[4, 0].scatter([x], [y], s=30, c="r", marker="*")
    g.subplots[0, 4].scatter([y], [x], s=30, c="r", marker="*")

    for i, param in enumerate(useparams):
        idx = case_MAP.allparams.index(param)
        mu = case_MAP.values_ML[idx]
        sigma = np.sqrt(case_MAP.invfshp[idx, idx])
        x = np.linspace(mu-4*sigma, mu+4*sigma, 201)
        g.subplots[i, i].plot(x, norm.pdf(x, loc=mu, scale=sigma) \
            / norm.pdf(mu, loc=mu, scale=sigma), "C1", ls="--", lw=0.5)

        # 1D ML curves, 5/5/2026.
        norm_MAP = norm.pdf(mu, loc=mu, scale=sigma)
        mu = case_ML.values_ML[idx]
        sigma = np.sqrt(case_ML.invfsh[idx, idx])
        x = np.linspace(mu-4*sigma, mu+4*sigma, 201)
        g.subplots[i, i].plot(x, norm.pdf(x, loc=mu, scale=sigma) \
            / norm_MAP, "C0", ls="-.", lw=0.5)

        g.subplots[i, 0].set_ylabel(labels.get(param, param))
        g.subplots[-1, i].set_xlabel(labels.get(param, param))

    if len(useparams) == 13:
        g.export(f"fisher_plots/corner_{space}.pdf")
    else:
        g.export(f"fisher_plots/corsub_{space}.pdf")

In [11]:
for space in ["real", "fourier"]:
    for useparams in [params, subparams]:
        corner_plots(space, vars()[f"mcmc_{space}"], useparams)

# Fisher and Parameter Covariance Matrices (Figures 4 and 7)

In [12]:
# Figure 4: Fisher Matrices
fig, axs = plt.subplots(2, 3, figsize=(10.8, 6.0), sharex=True, sharey=True)

for j, meta in enumerate([fourier, real]):
    base = FisherCase(meta)
    base.compute_values_and_errors(store_mats=True)
    vars()[f"invfshp_{meta.space}"] = base.invfshp

    for i, (mat, name) in enumerate(zip(["fisher", "invfsh", "invfshp"],
                                        ["Fisher", "Inverse (w/o Priors)", "Inverse (w/ Priors)"])):
        ax = axs[j, i]
        show_matrix(fig, ax, getattr(base, mat), symlog=True, oom=7)
        ax.set_title(f"{meta.space.capitalize()}: {name}")

        for tick in [5, 13, 21, 23]:
            ax.axhline(tick-0.5, color="k", lw=1, ls="--")
            ax.axvline(tick-0.5, color="k", lw=1, ls="--")

        for axis in ["x", "y"]:
            getattr(ax, f"set_{axis}ticks")(np.array([5, 13, 21, 23])-0.5, labels=[])
            getattr(ax, f"set_{axis}ticks")([2, 8.5, 16.5, 21.5, 26] if axis == "y"
                                            else [0, 8.5, 16.5, 21.5, 27], minor=True)
            getattr(ax, f"set_{axis}ticklabels")(["Cosmo-\nlogy", "Photo-$z$\nbias",
                                                "Shear\nbias", "IA", "Galaxy\nbias"], minor=True)
            getattr(ax, f"tick_params")(axis=axis, which="minor", length=0)

# fig.tight_layout()
# plt.show()
fig.savefig(f"fisher_plots/fisher.pdf", bbox_inches="tight")
plt.close(fig)

In [13]:
allparams = FisherMeta.ALLPARAMS
allparams[0] = r"$\sigma_8$"
allparams[2] = "h0"

def get_covmat(samples):
    covmat_ = samples.getCovMat()

    covmat = np.zeros((len(allparams),)*2)
    for i, param_i in enumerate(allparams):
        index_i = covmat_.paramNames.index(param_i)
        for j, param_j in enumerate(allparams):
            index_j = covmat_.paramNames.index(param_j)
            covmat[i, j] = covmat_.matrix[index_i, index_j]

    return covmat

covmat_real = get_covmat(mcmc_real[0])
covmat_fourier = get_covmat(mcmc_fourier[0])

In [14]:
def get_fommat(covmat):
    fommat = np.zeros_like(covmat)

    for i in range(len(allparams)):
        fommat[i, i] = 1 / covmat[i, i]
        for j in range(i+1, len(allparams)):
            fommat[i, j] = fommat[j, i] =\
                1 / np.sqrt(np.linalg.det(covmat[np.ix_([i, j], [i, j])]))

    return fommat

In [15]:
# Figure 7: Covariance Matrices of Parameters
fig, axs = plt.subplots(2, 3, figsize=(10.8, 6.0))

g = plots.get_single_plotter()
for j, space in enumerate(["fourier", "real"]):
    g.plot_2d([vars()[f"mcmc_{space}"][0]], r"$\sigma_8$", "omegam",
              shaded=True, ax=axs[j, 0], colors=["C4"])
    axs[j, 0].set_title(f"{space.capitalize()}: Main FoMs")
    axs[j, 0].set_ylabel(r"$\Omega_{\rm m}$")

    plot_mu_and_cov(vars()[f"casemap_{space}"], "MAP", "sigma8", "omegam", axs[j, 0],
                    "C1", marker="+", s=50, n_std=n_std, ls="--", lw=1, zorder=5)
    axs[j, 0].axvline(globals()[f"truth_{space}"]["sigma8"], color="grey", lw=1, ls=":", zorder=3)
    axs[j, 0].axhline(globals()[f"truth_{space}"]["omegam"], color="grey", lw=1, ls=":", zorder=3)

    covmat, invfshp = vars()[f"covmat_{space}"], vars()[f"invfshp_{space}"]
    print(space, np.linalg.slogdet(covmat), np.linalg.slogdet(invfshp))
    show_matrix(fig, axs[j, 1], covmat, symlog=True, oom=7)
    axs[j, 1].set_title(f"{space.capitalize()}: Cov(MCMC)")
    fomfshp, fommcmc = get_fommat(invfshp), get_fommat(covmat)
    show_matrix(fig, axs[j, 2], fomfshp / fommcmc - 1)
    axs[j, 2].set_title(f"{space.capitalize()}: FoM(Fisher)/FoM(MCMC)-1")

    for c, ls, fommat, name in zip(["C1", "C4"], ["--", "-"],
                                   [fomfshp, fommcmc], ["Fisher", "MCMC"]):
        axs[j, 0].plot([], [], c=c, ls=ls, label=f"{name}:\n" +\
            f"FoM1={fommat[0, 0]:.2e}\nFoM2={fommat[0, 4]:.2e}".replace("+0", ""))
    h, l = axs[j, 0].get_legend_handles_labels()
    axs[j, 0].add_artist(axs[j, 0].legend(h[:1], l[:1], loc="upper right"))
    axs[j, 0].legend(h[1:], l[1:], loc="lower left")

    for i in range(1, 3):
        ax = axs[j, i]
        for tick in [5, 13, 21, 23]:
            ax.axhline(tick-0.5, color="k", lw=1, ls="--")
            ax.axvline(tick-0.5, color="k", lw=1, ls="--")

        for axis in ["x", "y"]:
            getattr(ax, f"set_{axis}ticks")(np.array([5, 13, 21, 23])-0.5, labels=[])
            getattr(ax, f"set_{axis}ticks")([2, 8.5, 16.5, 21.5, 26] if axis == "y"
                                            else [0, 8.5, 16.5, 21.5, 27], minor=True)
            getattr(ax, f"set_{axis}ticklabels")(
                ["Cosmo-\nlogy", "Photo-$z$\nbias", "Shear\nbias", "IA", "Galaxy\nbias"]
                if j == 1 and axis == "x" or i == 1 and axis == "y" else [], minor=True)
            getattr(ax, f"tick_params")(axis=axis, which="minor", length=0)

axs[0, 0].set_xlabel("")
fig.tight_layout()
fig.savefig(f"fisher_plots/fisher_cf.pdf", bbox_inches="tight")
plt.close(fig)

fourier SlogdetResult(sign=1.0, logabsdet=-374.1909584709265) SlogdetResult(sign=1.0, logabsdet=-377.8125290293144)
real SlogdetResult(sign=1.0, logabsdet=-340.26519222386327) SlogdetResult(sign=1.0, logabsdet=-339.7000726900535)


In [16]:
# Figure 7: Covariance Matrices of Parameters
fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.6))

g = plots.get_single_plotter()
for j, space in enumerate(["fourier", "real"]):
    g.plot_2d([vars()[f"mcmc_{space}"][0]], r"$\sigma_8$", "omegam",
              shaded=True, ax=axs[j], colors=["C4"])
    axs[j].set_title(f"{space.capitalize()}: Main FoMs")
    axs[j].set_xlabel(r"$\sigma_8$")

    plot_mu_and_cov(vars()[f"casemap_{space}"], "MAP", "sigma8", "omegam", axs[j],
                    "C1", marker="+", s=50, n_std=n_std, ls="--", lw=1, zorder=5)
    axs[j].axvline(globals()[f"truth_{space}"]["sigma8"], color="grey", lw=1, ls=":", zorder=3)
    axs[j].axhline(globals()[f"truth_{space}"]["omegam"], color="grey", lw=1, ls=":", zorder=3)

    covmat, invfshp = vars()[f"covmat_{space}"], vars()[f"invfshp_{space}"]
    fomfshp, fommcmc = get_fommat(invfshp), get_fommat(covmat)
    for c, ls, fommat, name in zip(["C1", "C4"], ["--", "-"],
                                   [fomfshp, fommcmc], ["Fisher", "MCMC"]):
        axs[j].plot([], [], c=c, ls=ls, label=f"{name}:\n" +\
            f"FoM1={fommat[0, 0]:.2e}\nFoM2={fommat[0, 4]:.2e}".replace("+0", ""))
    h, l = axs[j].get_legend_handles_labels()
    axs[j].add_artist(axs[j].legend(h[:1], l[:1], loc="upper right"))
    axs[j].legend(h[1:], l[1:], loc="lower left")

axs[0].set_ylabel(r"$\Omega_{\rm m}$")
axs[1].set_ylabel("")
fig.tight_layout()
fig.savefig(f"fisher_plots/fisher_mcmc.pdf", bbox_inches="tight")
plt.close(fig)